---
---

# Lab 4: Great British Bake Off (A/B Test)

Welcome to Lab 4! Section 4's lab will focus on A/B Testing using data from the ever-popular British television show, [*The Great British Bake Off*](https://en.wikipedia.org/wiki/The_Great_British_Bake_Off).

#### **Helpful Resource:**
- [Python Reference](https://ulwazi.wits.ac.za/courses/89081/pages/detailed-python-reference-sheet-python-cheat-sheet-2?module_item_id=1200407)

**Recommended Readings:**

* [Error Probabilities](https://inferentialthinking.com/chapters/11/4/Error_Probabilities.html)
* [A/B Testing](https://inferentialthinking.com/chapters/12/1/AB_Testing.html)

**Submission**: Once you’re finished, run all cells besides the last one, select File > Save Notebook, or click the download button to download the .ipynb file. **Please save your work before downloading!**

<img src="download_ipynb.png" alt="download ipynb" width="800"/>

Let's begin by setting up the tests and imports by running the cell below.

In [ ]:
# Run this cell to set up the notebook, but please don't change it.
%pip install -q urllib3<2.0 otter-grader datascience ipywidgets
try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

import numpy as np
from datascience import *

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plots
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab4.ipynb")

---
---

## 1. A/B Testing

A/B testing is a form of hypothesis testing that allows you to make comparisons between two distributions — distribution of group “A” and distribution of group “B”. We may also refer to an A/B test as a permutation test.

You'll almost never be explicitly asked to perform an A/B test. Make sure you can identify situations where the test is appropriate and know how to correctly implement each step. Oftentimes, we use an A/B test to determine whether or not two samples came from the same underlying distribution.

---
---

**Question 1.1.** The following statements are the steps of an A/B hypothesis test presented in a *random order*:

1. Choose a test statistic (typically the difference in means between two categories)

2. Shuffle the labels of the original sample, find your simulated test statistic, and repeat many times

3. Find the value of the observed test statistic

4. Calculate the p-value based off your observed and simulated test statistics

5. Define a null and alternate hypothesis

6. Use the p-value and p-value cutoff to draw a conclusion about the null hypothesis

Assign `ab_test_order` to an array of integers that contains the correct order of an A/B test, where the first item of the array is the first step of an A/B test and the last item of the array is the last step of an A/B test.


In [ ]:
ab_test_order = ...

In [ ]:
grader.check("q1_1")

---
---

## 2. The Great British Bake Off

>"The Great British Bake Off (often abbreviated to Bake Off or GBBO) is a British television baking competition, produced by Love Productions, in which a group of amateur bakers compete against each other in a series of rounds, attempting to impress a group of judges with their baking skills" [Wikipedia](https://en.wikipedia.org/wiki/The_Great_British_Bake_Off)

For every week of the competition, the judges assign one contestant the title "Star Baker". Ultimately, one winner is crowned every season. Using this information, we would like to investigate how winning Star Baker awards affects the odds of winning a season of the show.

---
---

### Running an Experiment

We are going to run the following hypothesis test to determine the association between winning and number of Star Baker awards. The population we are examining is every contestant from seasons 2 through 11 of GBBO. We are going to use the following null and alternative hypotheses:

**Null hypothesis:** The distribution of Star Baker awards between contestants who won their season and contestants who did not win their season is the same.

**Alternative hypothesis:** Contestants who win their season of the show will win more Star Baker awards on average.

Our alternative hypothesis is related to our suspicion that contestants who win more Star Baker awards are more skilled, so they are more likely to win the season.

The `bakers` table below describes the number of star baker awards each contest won and whether or not they won their season (`1` if they won, `0` if they did not win). The data was manually aggregated from Wikipedia for seasons 2-11 of the show. We randomized the order of rows as to not spoil the outcome of the show.

In [ ]:
bakers = Table.read_table("star_bakers.csv")
bakers.show(3)

---
---

**Question 2.1.** Create a new table called `means` that contains the mean number of star baker awards for bakers who did not win (`won==0`) and bakers that did win (`won==1`). The table should have the column names `won` and `star baker awards mean`.

In [ ]:
means = ...
means

In [ ]:
grader.check("q2_1")

---

### The two distributions

Run the cell below to see the distribution of Star Baker awards for winners and
non-winners, drawn as overlaid histograms. Look at where the two shapes sit
relative to each other before moving on - the next question asks you to choose a
test statistic, and the answer should follow from what you see here.

In [ ]:
# Just run this cell
useful_bins = np.arange(0, 7)
bakers.hist('star baker awards', group='won', bins=useful_bins)

---
---

**Question 2.2.** We want to figure out if there is a difference between the distribution of Star Baker awards between winners and non winners. 

What should the test statistic be? Which values of this test statistic support the null, and which values support the alternative? **Assign `test_option` to the number corresponding to the correct test statistic.**

1. Absolute value of the difference between the means between both groups; high values support the null
2. Absolute value of the difference between the means between both groups; low values support the null
3. Average Star Baker awards for winners - average Star Baker awards for non-winners; high values support the null
4. Average Star Baker awards for winners - average Star Baker awards for non-winners; low values support the null

Before moving on, confirm your answer with a peer or in the discussion forums.

_Hint:_ You should think about what measures we use to describe a distribution. 


In [ ]:
test_option = ...

In [ ]:
grader.check("q2_2")

---
---

**Question 2.3.** Set `observed_difference` to the observed test statistic using the `means` table. 


In [ ]:
observed_difference = ...
observed_difference

In [ ]:
grader.check("q2_3")

---
---

**Question 2.4.** Given a table like `bakers`, a label column `label_col`, and a values column `val_col`, write a function that calculates the appropriate test statistic.

*Hint:* Make sure that you are taking the directionality of our alternative hypothesis into account.


In [ ]:
def find_test_stat(tbl, label_col, val_col):
    ...

find_test_stat(bakers, "won", "star baker awards")

In [ ]:
grader.check("q2_4")

When we run a simulation for A/B testing, we resample by **shuffling the labels** of the original sample. If the null hypothesis is true and the star baker award distributions are the same, we expect that the difference in mean star baker awards to not change when `"won"` labels are changed.

---
---

**Question 2.5.** Write a function `simulate_and_test_statistic` to compute one trial of our A/B test. Your function should run a simulation and return a test statistic.

*Hint:* Textbook chapter [12.1.4](https://inferentialthinking.com/chapters/12/1/AB_Testing.html#predicting-the-statistic-under-the-null-hypothesis) can help!

In [ ]:
def simulate_and_test_statistic(tbl, labels_col, values_col):
    ...

simulate_and_test_statistic(bakers, "won", "star baker awards")

In [ ]:
grader.check("q2_5")

---
---

**Question 2.6.** Simulate 5000 trials of our A/B test and store the test statistics in an array called `differences`.


In [ ]:
# This cell might take a couple seconds to run
differences = make_array()

...
                                                 
differences

In [ ]:
grader.check("q2_6")

Run the cell below to view a histogram of your simulated test statistics plotted with your observed test statistic as a red dot.

In [ ]:
Table().with_column('Difference Between Group Means', differences).hist(bins=20)
plots.scatter(observed_difference, 0, color='red', s=30, zorder=2)
plots.ylim(-0.1, 1.35);

---
---

**Question 2.7.** Find the p-value for your test and assign it to `empirical_p`.


In [ ]:
empirical_p = ...
empirical_p

In [ ]:
grader.check("q2_7")

---

### What the p-value tells you

Your p-value is well below 5%, so the data are not consistent with the null
hypothesis. Contestants who won their season really did earn more Star Baker
awards than those who did not.

Be careful about what that does **not** show. The test compares two groups that
were not assigned by anyone - it says the association is unlikely to be chance,
and nothing at all about whether the awards *caused* the wins. That distinction
is the subject of this session.

*Hint:* See section [12.1.6](https://inferentialthinking.com/chapters/12/1/AB_Testing.html#conclusion-of-the-test)


---

## You're done!

**Important submission information:**
- **Run all the tests** and verify that they all pass
- **Save** from the **File** menu
- **Run the final cell to generate the grader.check_all()**
- **Click the download button to download the .ipynb file (see the figure below)**
- Then, go to Ulwazi and submit the .ipynb file to Lab 4: Great British Bake Off (A/B Test).

**It is your responsibility to make sure your work is saved before running the last cell.**


<img src="download_ipynb.png" alt="download ipynb" width="800"/>

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order clicking the download button to download the .ipynb file. **Please save your work before downloading!**

<img src="download_ipynb.png" alt="download ipynb" width="800"/>